# S6E10 — 10 RealMLP on the rich feature set

> **Pipeline position:** 07 RealMLP → 08 TabPFN → 09 `X_rich` → **10 RealMLP-rich** → 11 stack

| | |
|---|---|
| **Input** | `data/processed/X_rich_*.parquet` (from `09`), `y_train.parquet` |
| **Output** | `preds/oof_realmlp_rich[_s1,_s2].npy`, `test_realmlp_rich*.npy`, averaged `oof_realmlp_rich_avg.npy` / `test_realmlp_rich_avg.npy` |
| **Runtime** | ≈ 5 min per fold with 4 networks on the Mac M4 CPU (~25 min per seed) |
| **Result** | OOF AUC **0.96122** per seed, **0.96129** averaged over 3 seeds (RealMLP on `X_final`: 0.96082) |

**Why:** `07` gave RealMLP on `X_final` OOF 0.96075 (single seed) / 0.96082 (3 seeds). The public notebook gets 0.96123 with extra feature families. `09` built them. Here the *same recipe as `07`* runs on the rich set, plus **in-fold target encoding** of 7 keys (route, age, two digit variants, three combined keys), fitted on the training part of each fold only. The only change versus `07` is the input, so any CV gain is attributable to the features.

**Recipe:** identical to `07` (RealMLP-TD community settings: categorical twins, PLR embeddings, lr 0.053 with schedule, 512-256-128, SiLU, 4 epochs, several networks per model). See `07` for the explanation of every ingredient.

**Input size:** 103 columns after `nn_frame` (92 − `orig_prob` − 3 `key_*` + 15 categorical twins) + 7 target-encoded columns per fold.

> **Glossary:** *Seed* = random initialisation of a network; different seeds give slightly different networks, averaging them reduces noise. *In-fold target encoding* = the satisfied share per key, fitted on the training part of each fold only (see `02` E12). *Categorical twin*, *PLR embedding* = see `07`.

**Plan:** (1) timing probe; (2) fold 0 as benchmark — RealMLP on `X_final` had **0.96092**; gain < +0.0002 on one fold is noise; (3) all folds, resumable; then more seeds, averaged.

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import sys, subprocess, importlib.util
if importlib.util.find_spec('pytabkit') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytabkit'], check=False)
import torch
import numpy as np, pandas as pd, time, gc, warnings
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
from pytabkit import RealMLP_TD_Classifier
warnings.filterwarnings('ignore')

SMOKE = False        # True: 60k training rows, 1 epoch, 2 networks: only checks that the code runs
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'      # 'mps' is not used (unstable in this environment)
N_ENS = 8 if DEVICE == 'cuda' else 4     # networks per model; 8 in the source notebook, 4 on CPU to keep the time acceptable
N_EPOCHS = 4
SEED = 2            # run the notebook again with SEED = 1, 2, ... to get more seeds (seed 0 is already done); they are averaged in section 6
print('device:', DEVICE, '| networks per model:', N_ENS, '| epochs:', N_EPOCHS)

N_FOLDS, RANDOM_STATE = 5, 42
CANDS = [Path('data/processed'), Path('../data/processed'), Path('/kaggle/working'), Path('.')]
PROC_DIR = next(p for p in CANDS if (p / 'X_rich_train.parquet').exists())
PREDS_DIR = next((p for p in [Path('preds'), Path('../preds'), Path('/kaggle/working/preds')] if p.exists()), Path('preds')); PREDS_DIR.mkdir(exist_ok=True)
X = pd.read_parquet(PROC_DIR / 'X_rich_train.parquet'); X_test = pd.read_parquet(PROC_DIR / 'X_rich_test.parquet')
y = pd.read_parquet(PROC_DIR / 'y_train.parquet')['y'].astype(int).values
FOLDS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE).split(X, y))
assert list(X.columns) == list(X_test.columns)
print(X.shape, X_test.shape, '| token columns:', sum(c.startswith('tok_') or 'tok' in c for c in X.columns))

/opt/anaconda3/envs/kaggle-playground/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


device: cpu | networks per model: 4 | epochs: 4
(699635, 92) (299844, 92) | token columns: 12


## 1. Input frame
`nn_frame` builds the RealMLP input: rich columns, `orig_prob` dropped (`orig_logit` kept), categorical twins for route, age and the 13 ratings. The `key_*` columns are dropped (they only serve target encoding). The target-encoded `te_*` columns are added per fold by `add_te`. Category lists come from train+test together (fixed), so no fold meets an unknown level.

In [2]:
RATINGS = ['inflight_wifi_service','departure/arrival_time_convenient','ease_of_online_booking','gate_location','food_and_drink',
           'online_boarding','seat_comfort','inflight_entertainment','on-board_service','leg_room_service','baggage_handling',
           'checkin_service','cleanliness']
TWINS = {'route_id': 'flight_distance', 'age_cat': 'age', **{f'{r}_cat': r for r in RATINGS}}
FULL = pd.concat([X, X_test], ignore_index=True)
TWIN_LEVELS = {t: sorted(FULL[s].astype(int).unique()) for t, s in TWINS.items()}
# Integer ids used ONLY to build target encodings; not fed to the network as numbers.
KEYS = [c for c in X.columns if c.startswith('key_')]      # integer ids for target encoding only, not fed as numbers
# Columns that get an in-fold target encoding: route, age, two digit-based route variants, and the three combined keys.
TE_SOURCES = [c for c in ['flight_distance', 'age', 'fd_div10', 'fd_mod1000'] + KEYS if c in X.columns]
CAT_COLS = ['gender', 'customer_type', 'type_of_travel', 'class'] + list(TWINS)

def nn_frame(D):
    D = D.drop(columns=['orig_prob'] + KEYS).copy()
    for t, s in TWINS.items():
        D[t] = pd.Categorical(D[s].astype(int), categories=TWIN_LEVELS[t])
    assert not D.isna().any().any()
    return D

print('categorical columns:', len(CAT_COLS), '| route levels:', len(TWIN_LEVELS['route_id']), '| input columns:', nn_frame(X.iloc[:5]).shape[1])

categorical columns: 19 | route levels: 3521 | input columns: 103


## 2. Parameters (the community recipe)

In [3]:
PARAMS = dict(
    n_epochs=1 if SMOKE else N_EPOCHS, n_ens=2 if SMOKE else N_ENS, batch_size=256, use_early_stopping=False,
    val_metric_name='1-auc_ovr', device=DEVICE, random_state=0, verbosity=0,
    lr=0.053, wd=0.0236, sq_mom=0.988, lr_sched='lin_cos_log_15', wd_sched='cos_log_15',
    first_layer_lr_factor=0.25, embedding_size=5, max_one_hot_cat_size=18,
    hidden_sizes=[512, 256, 128], act='silu', p_drop=0.05, p_drop_sched='expm4t',
    plr_hidden_1=16, plr_hidden_2=8, plr_act_name='gelu', plr_lr_factor=0.1151, plr_sigma=2.33,
    use_ls=True, ls_eps=0.01, ls_eps_sched='sqrt_cos', add_front_scale=False,
    bias_init_mode='neg-uniform-dynamic-2',
    tfms=['one_hot', 'median_center', 'robust_scale', 'smooth_clip', 'embedding', 'l2_normalize'],
)

# Why in-fold: the encoding uses labels, so it is fitted on the fold's TRAINING part (inner cv=5 so training rows are not encoded with their own label) and merely applied to validation/test.
def add_te(Xtr, ytr, others):
    # in-fold target encoding: fit on the training part (internal 5-fold cross-fitting for the training rows), transform the rest
    te = TargetEncoder(target_type='binary', cv=5, shuffle=True, random_state=42)
    cols = [f'te_{c}' for c in TE_SOURCES]
    tr_te = pd.DataFrame(te.fit_transform(Xtr[TE_SOURCES].astype(int), ytr), columns=cols, index=Xtr.index).astype('float32')
    outs = [pd.DataFrame(te.transform(o[TE_SOURCES].astype(int)), columns=cols, index=o.index).astype('float32') for o in others]
    return [pd.concat([D, T], axis=1) for D, T in zip([Xtr] + list(others), [tr_te] + outs)]

# Pipeline per fold: add TE -> nn_frame (drop keys, add categorical twins) -> RealMLP -> predict validation (OOF) and test.
def fit_predict(tr_idx, va_idx, seed=None, rows=None):
    seed = SEED if seed is None else seed
    tr_idx = tr_idx if rows is None else tr_idx[:rows]
    Xtr, Xva, Xte = add_te(X.iloc[tr_idx], y[tr_idx], [X.iloc[va_idx], X_test])
    Xtr, Xva, Xte = nn_frame(Xtr), nn_frame(Xva), nn_frame(Xte)
    model = RealMLP_TD_Classifier(**{**PARAMS, 'random_state': seed})
    model.fit(Xtr, y[tr_idx], X_val=Xva, y_val=y[va_idx], cat_col_names=CAT_COLS)
    p_va, p_te = model.predict_proba(Xva)[:, 1], model.predict_proba(Xte)[:, 1]
    del model; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    return p_va, p_te

## 3. Timing probe
One epoch-equivalent on 50,000 rows, to extrapolate the time of a full fold (≈ 560,000 rows x 4 epochs). Decide from this where to run the rest (this machine, the home GPU, Kaggle GPU).

In [4]:
tr0, va0 = FOLDS[0]
t0 = time.time()
old = PARAMS['n_epochs']; PARAMS['n_epochs'] = 1
_ = fit_predict(tr0, va0[:20000], rows=50000)
PARAMS['n_epochs'] = old
t_probe = time.time() - t0
est = t_probe * (len(tr0) / 50000) * (old if not SMOKE else 1)
print(f'probe: {t_probe:.0f} s for 50k rows x 1 epoch (includes setup and prediction of 20k + 300k test rows)')
print(f'rough estimate for one full fold: {est/60:.0f} min -> 5 folds: {5*est/3600:.1f} h  (an upper bound: setup and test prediction are included in the probe)')

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer.fit` stopped: `max_epochs=1` reached.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


probe: 21 s for 50k rows x 1 epoch (includes setup and prediction of 20k + 300k test rows)
rough estimate for one full fold: 16 min -> 5 folds: 1.3 h  (an upper bound: setup and test prediction are included in the probe)


## 4. Fold 0 — benchmark
Compare with RealMLP on `X_final` (fold 0: **0.96092**). Gain below +0.0002 would be noise-level for a single fold.

In [5]:
t0 = time.time()
p_va0, p_te0 = fit_predict(*FOLDS[0], rows=60000 if SMOKE else None)
auc0 = roc_auc_score(y[FOLDS[0][1]], p_va0)
print(f'fold 0: RealMLP-rich AUC = {auc0:.5f}  ({(time.time()-t0)/60:.1f} min)   | RealMLP on X_final 0.96092')

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer.fit` stopped: `max_epochs=4` reached.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


fold 0: RealMLP-rich AUC = 0.96133  (5.0 min)   | RealMLP on X_final 0.96092


## 5. All 5 folds (resumable)
Each finished fold is saved in `preds/realmlp_rich_folds/` (further seeds: `realmlp_rich_folds_s1`, `_s2`). Run this cell after reading the benchmark above; if the run is interrupted, running it again continues from the last finished fold.
Set `RUN_ALL = False` to skip it (e.g. while only the benchmark matters).

In [6]:
RUN_ALL = True
if RUN_ALL and not SMOKE:
    CK = PREDS_DIR / ('realmlp_rich_folds' if SEED == 0 else f'realmlp_rich_folds_s{SEED}'); CK.mkdir(exist_ok=True)
    oof = np.full(len(X), np.nan); tests = []
    for k, (tr, va) in enumerate(FOLDS):
        fo, ft = CK / f'oof_{k}.npy', CK / f'test_{k}.npy'
        if fo.exists() and ft.exists():
            oof[va] = np.load(fo); tests.append(np.load(ft)); print(f'fold {k}: loaded'); continue
        # Fold 0 was already trained in the benchmark cell; reuse it.
        if k == 0:                                   # already computed above
            p_va, p_te = p_va0, p_te0
        else:
            t0 = time.time(); p_va, p_te = fit_predict(tr, va)
        oof[va] = p_va; tests.append(p_te); np.save(fo, p_va); np.save(ft, p_te)
        print(f'fold {k}: AUC {roc_auc_score(y[va], p_va):.5f}', flush=True)
    print(f'\nOOF AUC (RealMLP rich): {roc_auc_score(y, oof):.5f}')
    sfx = '' if SEED == 0 else f'_s{SEED}'
    np.save(PREDS_DIR / f'oof_realmlp_rich{sfx}.npy', oof); np.save(PREDS_DIR / f'test_realmlp_rich{sfx}.npy', np.mean(tests, axis=0))
    print('saved', PREDS_DIR / f'oof_realmlp_rich{sfx}.npy', '(probabilities)')

fold 0: AUC 0.96133


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer.fit` stopped: `max_epochs=4` reached.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


fold 1: AUC 0.96026


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer.fit` stopped: `max_epochs=4` reached.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


fold 2: AUC 0.96205


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer.fit` stopped: `max_epochs=4` reached.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


fold 3: AUC 0.96131


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer.fit` stopped: `max_epochs=4` reached.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


fold 4: AUC 0.96120

OOF AUC (RealMLP rich): 0.96122
saved preds/oof_realmlp_rich_s2.npy (probabilities)


## 6. Average the seeds
Averages the logits of all seeds found in `preds/` (seed 0 = `oof_realmlp_rich.npy`, further seeds = `oof_realmlp_rich_s1.npy`, ...) and writes `oof_realmlp_rich_avg.npy` / `test_realmlp_rich_avg.npy`. `11_stack` uses the averaged files automatically when they exist.

In [7]:
if not SMOKE:
    # Seeds averaged on the logit scale (see 07).
    lg = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    sig = lambda z: 1 / (1 + np.exp(-z))
    seeds = [(0, PREDS_DIR / 'oof_realmlp_rich.npy', PREDS_DIR / 'test_realmlp_rich.npy')] + [(s, PREDS_DIR / f'oof_realmlp_rich_s{s}.npy', PREDS_DIR / f'test_realmlp_rich_s{s}.npy') for s in range(1, 10)]
    seeds = [t for t in seeds if t[1].exists() and t[2].exists()]
    O_ = [lg(np.load(o)) for _, o, _ in seeds]; T_ = [lg(np.load(t)) for _, _, t in seeds]
    for (s, _, _), o in zip(seeds, O_): print(f'seed {s}: OOF AUC {roc_auc_score(y, o):.5f}')
    o_avg, t_avg = np.mean(O_, axis=0), np.mean(T_, axis=0)
    print(f'average of {len(seeds)} seeds: OOF AUC {roc_auc_score(y, o_avg):.5f}')
    if len(seeds) > 1:
        np.save(PREDS_DIR / 'oof_realmlp_rich_avg.npy', sig(o_avg)); np.save(PREDS_DIR / 'test_realmlp_rich_avg.npy', sig(t_avg)); print('saved the averaged files')

seed 0: OOF AUC 0.96117
seed 1: OOF AUC 0.96125
seed 2: OOF AUC 0.96122
average of 3 seeds: OOF AUC 0.96129
saved the averaged files


## Results & decisions

| Run | OOF AUC | Fold AUCs |
|---|---|---|
| fold 0 vs `X_final` | 0.96133 vs 0.96092 | **+0.00041** |
| seed 0 / 1 / 2 | 0.96117 / 0.96125 / 0.96122 | seed 2: 0.96133 / 0.96026 / 0.96205 / 0.96131 / 0.96120 |
| **mean of 3 seeds (logits)** | **0.96129** | |
| *RealMLP on `X_final` (07), 3 seeds* | *0.96082* | |

- **The richer features help: +0.00047 OOF** over `07` with the identical recipe. This is far larger than seed noise (the three seeds span only 0.00008).
- **Which family did the work is only partly isolated:** the GPT-2 token block adds nothing (fold 0: 0.96134 with 92 columns vs 0.96133 with 80; the 80-column run is saved as `oof_realmlp_rich80.npy`). Digits, delays, original-survey rates, the second captain and the TE keys were added together; a per-family ablation was not run (the later experiments `12`–`22` went for new information instead of pruning).
- **Role in the stack:** `realmlp_rich` gets the second-largest weight (0.39, after TabPFN's 0.66) — see `11`.
- **Practical cost:** CPU time is the limiting factor (≈ 25 min per seed on the Mac); 8 networks and more seeds are better done on the GPU.

## Next steps
`11_stack.ipynb` has the model `realmlp_rich` (it takes the averaged files when they exist). Rerun it and look at the nested-CV table: keep it only if it beats the stack without it.